In [1]:
from bs4 import BeautifulSoup
import requests, os, csv, urllib.parse, urllib3, time, random
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

import numpy as np
import pandas as pd
from tqdm import tqdm
import re,os,glob, time,string
urllib3.disable_warnings()

import web_general
from IPython.display import Image

import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

In [2]:
chrome_options = Options()
chrome_options.add_argument("--headless")

In [3]:
url='http://www.npo.gov.za/PublicNpo/Npo'
driver = webdriver.Chrome("chromedriver.exe") 
driver.get(url) 
time.sleep(5)

C:\Users\f3879844\AppData\Local\Temp/ipykernel_8108/3112341243.py:2: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome("chromedriver.exe")


In [4]:
def next_page():
    button = driver.find_element(By.ID, "next_PublicNpoSearch_Index-Pager")
    button.click()

def get_details_save(rows_,count,df):
    for row in rows_[1:]:
        hold_list=[]
        for item in row:
            hold_list.append(item.text)
        hold_list.append(count)
        df.loc[len(df)] = hold_list
    return df

def get_rows(driver):
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table=soup.find(lambda tag: tag.name=='table' and tag.has_attr('id') and tag['id']=="PublicNpoSearch_Index-Table") 
    rows = table.findAll(lambda tag: tag.name=='tr')
    page_view=soup.find("div", class_="scroll ui-state-default ui-jqgrid-pager ui-corner-bottom").text
    return rows,page_view

def get_details(rows_,count):
    for row in rows_[1:]:
        hold_list=[]
        for item in row:
            hold_list.append(item.text)
        hold_list.append(count)
        save_list=hold_list
        last_name=hold_list[4]
    return save_list, last_name

In [5]:
page_num_df=0 #11324 #8617#2651#df.iloc[-1,df.columns.get_loc("Page")]
check=0
count=0
while (page_num_df <= 3):
    
    #First time it comes here it goes to next page
    if check==0:
        next_page()
        
    rows,page_view=get_rows(driver)
    page_view=int(int(page_view.split("-")[1].split("of")[0].replace(" ", ""))/20)
    
    if page_num_df!=page_view:
        df = pd.DataFrame(columns=["ID","Search","Name","Province","Reg_no","DOR","Status","DC","FM","Page"])
        save_list, last_name=get_details(rows,count)
        df=get_details_save(rows,page_view,df)
        page_num_df=df.iloc[-1,df.columns.get_loc("Page")]
        print(page_num_df,end="\r")
        page_num_df=page_view
        
        df.to_csv(r"C:\Users\f3879844\OneDrive - FRG\Desktop\_temp\Shikash_Handover\Webscraping_General\Moved_Credit_Data\Testing\\"+str(page_num_df)+".csv",index=False) 
                    
        check=0
    else:
        check+=1
        time.sleep(0.2)
    
    if check >=30: #Keep checking if page updated.  If not updated force update
        next_page()
        check=1

In [6]:
# set working directory
os.chdir('testing')

In [7]:
#find all csv files in the folder
#use glob pattern matching -> extension = 'csv'
file_extension = 'csv'
#save result in list -> combined_files
combined_files = [i for i in glob.glob('*.{}'.format(file_extension))]
print(combined_files)

['1.csv', '2.csv', '3.csv', '4.csv']


In [8]:
#combine all files in the list
stacked_csv = pd.concat([pd.read_csv(j) for j in combined_files ])
#export to csv
stacked_csv.to_csv( "combined_output.csv", index=False, encoding='utf-8-sig')

In [9]:
# Load the data
df2 = pd.read_csv(r'combined_output.csv', encoding= 'unicode_escape', low_memory=False)
# Check the dimension of data
print('Dimension of data: {} rows and {} columns'.format(len(df), len(df.columns)))
# Print the first 5 rows
#df2.head()
df2.rename(columns = {'ï»¿ID':'ID'}, inplace = True)
df2

Dimension of data: 20 rows and 10 columns


,ID,Search,Name,Province,Reg_no,DOR,Status,DC,FM,Page
0,17207,NaN,"""Hearus""",Gauteng,017-207 NPO,28/01/2002,Registered,Â,Â,1
1,75845248,NaN,A ONE PUBLIC WELFARE FOUNDATION,Gauteng,255-972 NPO,24/02/2021,Registered,Â,Â,1
2,81453477,NaN,A RAY OF HOPE,Free State,264-685 NPO,02/09/2021,Registered,Â,Â,1
3,57210292,NaN,A RE BEREKENG BASADI,North West,227-303 NPO,06/05/2019,Registered,Â,Â,1
4,31592703,NaN,Abacebileyo Enkosini Richness In Christ Minis...,Kwaxuma,163-088 NPO,10/12/2015,Registered,Â,Â,1
...,...,...,...,...,...,...,...,...,...,...
75,73932308,NaN,Higher Education Facilities Management Associ...,Gauteng,254-848 NPO,02/02/2021,Registered,Â,Â,4
76,48212661,NaN,Hope Of VICTIMS,Gauteng,205-557 NPO,13/03/2018,Registered,Â,Â,4
77,52744134,NaN,House Of Bliss Tabernacle International,Standerton,218-393 NPO,13/11/2018,Registered,Â,Â,4
78,89921820,NaN,HOUSE OF CONQUEROR's,Western Cape,275-348 NPO,18/05/2022,Registered,Â,Â,4


In [10]:
list_urls = df2['ID'].values.tolist()

list_urls

[17207,
 75845248,
 81453477,
 57210292,
 31592703,
 42465560,
 53907545,
 47026010,
 67145222,
 67463274,
 51663948,
 48181963,
 38714761,
 47732908,
 45260341,
 81503777,
 68251971,
 15712,
 41310427,
 65171774,
 42875448,
 63048850,
 46262156,
 42928886,
 67762317,
 17163544,
 78604171,
 42142544,
 58535426,
 51489430,
 40570023,
 92225627,
 2259,
 40268643,
 87435502,
 42291657,
 42348115,
 63357514,
 42266489,
 79633958,
 55772990,
 49991586,
 50720282,
 48450633,
 58070697,
 91705988,
 38927391,
 38759169,
 55341773,
 59467640,
 52444767,
 53064537,
 73508535,
 66870,
 47897743,
 87860405,
 6853,
 90626698,
 67786558,
 55823595,
 44340368,
 39966871,
 74929585,
 37930843,
 57808486,
 43588473,
 79581941,
 81665985,
 21633568,
 62027742,
 60215391,
 49580193,
 20490,
 44051845,
 40953986,
 73932308,
 48212661,
 52744134,
 89921820,
 65588087]

Enter the URL of the website you will like to webscrape

In [11]:
URL='https://www.npo.gov.za/PublicNpo/Npo'

In [12]:
soup=web_general.parse_listing(web_general.buildSearchUrl(URL), web_general.get_proxies())
if not soup:print("Not found")
else: print("good to go")

good to go


We can now webscrape everything that we can see in the soup file.<br>

In [13]:
soup

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta content="noindex" name="robots"/>
<meta content="noindex" name="googlebot"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<title>
</title>
<!--// Stylesheets //-->
<link href="/Areas/PublicNpo/Content/jquery-ui-1.8.16.custom.css" rel="stylesheet" type="text/css"/>
<link href="/Areas/Shesha/Content/jqGrid/ui.jqgrid.css" rel="stylesheet" type="text/css"/>
<link href="/Areas/PublicNpo/Content/css/style.css" rel="stylesheet" type="text/css"/>
<link href="/Areas/PublicNpo/Content/css/color.css" rel="stylesheet" type="text/css"/>
<link href="/Areas/PublicNpo/Content/shesha.framework.css" rel="stylesheet" type="text/css"/>
<link href="/Areas/Shesha/Content/jquery.autocomplete.css" rel="stylesheet" type="text/css"/>
<link href="/Areas/Publi

In [14]:
base_url="https://www.npo.gov.za/PublicNpo/Npo/DetailsPublicDocs/"

In [15]:
for item in list_urls:
    item=str(item)
    search_url=base_url+item
    soup=web_general.parse_listing(web_general.buildSearchUrl(search_url), web_general.get_proxies())
    
    #print(soup.find("h1", class_="widget-head-border colr").text.split("\n")[1])
    name_list = soup.find("h1", class_="widget-head-border colr").text.split("\n")[1]
   

In [16]:
 name_list

'                NPO Details:  I CARE FOUNDATION'

In [ ]:
for item in list_urls:
    item=str(item)
    search_url=base_url+item
    soup=web_general.parse_listing(web_general.buildSearchUrl(search_url), web_general.get_proxies())
    
    #print(soup.find("div", class_="padbox").text.split("\n")[7])
    Reg_no_list = soup.find("div", class_="padbox").text.split("\n")[7]

In [ ]:
for item in list_urls:
    item=str(item)
    search_url=base_url+item
    soup=web_general.parse_listing(web_general.buildSearchUrl(search_url), web_general.get_proxies())
    
    print(soup.find("div", class_="padbox").text.split("\n")[8]) # [10] + [13]

In [ ]:
for item in list_urls:
    item=str(item)
    search_url=base_url+item
    soup=web_general.parse_listing(web_general.buildSearchUrl(search_url), web_general.get_proxies())
    
    #print(soup.find("div", class_="padbox").text.split("\n")[8]) # [10] + [13]
    a = soup.find("div", class_="padbox").text.split("\n")[8]
    b = a.split(">")

In [ ]:
print(a)
print(b)

In [ ]:
for item in list_urls:
    item=str(item)
    search_url=base_url+item
    soup=web_general.parse_listing(web_general.buildSearchUrl(search_url), web_general.get_proxies())
    
    print(soup.find("div", class_="padbox").text.split("\n")[10])

In [ ]:
for item in list_urls:
    item=str(item)
    search_url=base_url+item
    soup=web_general.parse_listing(web_general.buildSearchUrl(search_url), web_general.get_proxies())
    
    print(soup.find("div", class_="padbox").text.split("\n")[13])

In [ ]:
df = pd.DataFrame([list_urls],columns=["ID","Name","Reg_no","Cat1","Cat2","Cat3","NPO_Type","DOR"])
df = get_details_save(rows,page_view,df)

In [ ]:
df.to_csv("NPO_full_complete.csv")